# Second PyCUDA example: automatic data transfers

This is the second notebook in the five-example PyCUDA sequence. It computes the same vector sum as the first notebook, using the **same CUDA kernel, input values, numerical types, vector length and launch configuration**.

The change is in the Python host code: `cuda.In` and `cuda.Out` manage temporary device buffers and the copies associated with the kernel call. Run the cells in order in a GPU runtime. Results and timings appear directly below the cells; no result files are written.

## 1. Prepare the GPU runtime

In Google Colab, select **Runtime → Change runtime type → GPU** and connect. The following commands inspect the assigned GPU and CUDA compiler on the remote machine. `SourceModule` needs `nvcc`; a GPU runtime alone does not establish that the compiler is available.


In [1]:
!nvidia-smi
!nvcc --version


Wed Oct 14 05:52:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install PyCUDA if necessary. The installation may build the package and take several minutes. Continue only after installation succeeds.


In [2]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281396 sha256=0e593c710439be0034b9ed05e894597f5a948a0aff4be4b46bc180d6dff63411
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


## 2. Initialize PyCUDA

`pycuda.autoinit` creates the CUDA context used by this standalone example. `cuda` provides the driver operations; `SourceModule` compiles the kernel source into a module. CUDA errors are reported as Python exceptions.


In [3]:
import sys
import numpy as np
import pycuda
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

device = cuda.Context.get_device()
print('GPU:', device.name())
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyCUDA:', pycuda.VERSION_TEXT)


GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
PyCUDA: 2026.1


## 3. Choose the vector length and launch size

Use $N=100003$ and 256 threads per block, as in Chapter 2. The length is deliberately not divisible by the block size. `iDivUp(N, BLOCKSIZE)` computes the number of blocks by rounding the division upward.

The kernel uses 32-bit integer indexing. The checks below restrict the configuration to valid device limits and keep the padded launch indices within that range.


In [4]:
N = 100003
BLOCKSIZE = 256

def iDivUp(a, b):
    return (a + b - 1) // b

assert 1 <= BLOCKSIZE <= min(
    device.get_attribute(cuda.device_attribute.MAX_THREADS_PER_BLOCK),
    device.get_attribute(cuda.device_attribute.MAX_BLOCK_DIM_X),
)
assert 1 <= N <= np.iinfo(np.int32).max - (BLOCKSIZE - 1)

number_of_blocks = iDivUp(N, BLOCKSIZE)
assert number_of_blocks <= device.get_attribute(
    cuda.device_attribute.MAX_GRID_DIM_X
)
block = (BLOCKSIZE, 1, 1)
grid = (number_of_blocks, 1, 1)

print(f'Elements: {N}')
print(f'Blocks: {number_of_blocks}; threads per block: {BLOCKSIZE}')
print(f'Threads with no output element: {number_of_blocks * BLOCKSIZE - N}')


Elements: 100003
Blocks: 391; threads per block: 256
Threads with no output element: 93


## 4. Prepare the host arrays

The inputs reproduce the Chapter 2 initialization: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both are contiguous, one-dimensional `float32` arrays, matching CUDA's `float` pointers. `h_c` will receive the GPU result; `np.empty_like` allocates its storage without initializing the entries.


In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
h_c = np.empty_like(h_a)

assert h_a.dtype == h_b.dtype == h_c.dtype == np.float32
assert all(x.flags.c_contiguous for x in (h_a, h_b, h_c))
print(f'Bytes per vector: {h_a.nbytes}')


Bytes per vector: 400012


## 5. Compile the unchanged CUDA kernel

The kernel is identical to the first PyCUDA example and the Chapter 2 CUDA example. Its arguments remain `(a, b, c, n)`. `no_extern_c=True` avoids adding another linkage wrapper because the source already declares `extern "C"`.

In [6]:
kernel_source = r"""
extern "C" __global__
void vector_add(const float *a, const float *b, float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
"""

module = SourceModule(
    kernel_source, options=["-std=c++14"], no_extern_c=True
)
vector_add = module.get_function("vector_add")


## 6. Supply host arrays through argument handlers

The kernel still receives pointers to **device memory**. The following argument handlers tell PyCUDA how to prepare those pointers and move the data:

- `cuda.In(h_a)` and `cuda.In(h_b)` allocate device storage and copy the input arrays from host to device before execution.
- `cuda.Out(h_c)` allocates output storage and copies the result into the existing host array `h_c` after execution. It does not copy the initial contents of `h_c` to the GPU, since the kernel writes every valid output element.

This replaces the explicit allocation and copy calls in the first notebook. The launch dimensions and typed length are unchanged. With the default, no-explicit-stream invocation used here, the output copy has completed when the call returns, so `h_c` is ready for the following cells. PyCUDA manages the temporary device allocations; there are no named device buffers to free manually in this example.

In [7]:
vector_add(
    cuda.In(h_a), cuda.In(h_b), cuda.Out(h_c), np.int32(N),
    block=block, grid=grid,
)

## 7. Validate every output

Use the same NumPy reference and tolerances as in the first notebook. The final partial block is included in the check. No explicit device-to-host copy is needed here: `cuda.Out` has already filled `h_c`.

In [8]:
h_reference = h_a + h_b

if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f'Validation passed for all {N} elements.')
print(f'Maximum absolute error: {max_abs_error:.3e}')

Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 8. Display the computed sum

The `c[i] (GPU)` column comes from the output copied back by `cuda.Out`. As in the first notebook, display the first twelve and the last three entries.

In [9]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')


       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 9. Optional: measure the complete call

Each call below creates fresh argument handlers, manages temporary device storage, copies both inputs to the GPU, launches the kernel, and copies the output back. A host timer measures the **complete call**, including these operations and Python submission overhead.

After three warm-up calls, measure 100 repetitions and print the mean time per call. Compilation, input construction, validation and printing are outside the interval. Synchronization before starting the timer excludes earlier GPU work; the default `cuda.Out` path completes its output copy before each call returns.

This is **not kernel-only time**. It must not be compared directly with the first notebook's CUDA-event interval, which excludes allocation and transfers. A performance comparison between the methods would need to time the same operations in both.

In [10]:
from time import perf_counter

REPEATS = 100
assert REPEATS > 0

for _ in range(3):
    vector_add(
        cuda.In(h_a), cuda.In(h_b), cuda.Out(h_c), np.int32(N),
        block=block, grid=grid,
    )
cuda.Context.synchronize()

start = perf_counter()
for _ in range(REPEATS):
    vector_add(
        cuda.In(h_a), cuda.In(h_b), cuda.Out(h_c), np.int32(N),
        block=block, grid=grid,
    )
elapsed_ms = (perf_counter() - start) * 1000.0
mean_ms = elapsed_ms / REPEATS

# Check the last returned result outside the timed interval.
assert np.all(np.isfinite(h_c)), 'The GPU result contains nonfinite values.'
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
print(f'Repeated complete calls: {REPEATS}')
print(f'Total host elapsed time: {elapsed_ms:.6f} ms')
print(f'Mean complete-call time: {mean_ms:.6f} ms')
print('Includes temporary buffer management, input copies, kernel and output copy.')
print('Final result validated; this is not a kernel-only measurement.')

Repeated complete calls: 100
Total host elapsed time: 127.466479 ms
Mean complete-call time: 1.274665 ms
Includes temporary buffer management, input copies, kernel and output copy.
Final result validated; this is not a kernel-only measurement.


## What changed from the first PyCUDA example?

The numerical work on the GPU is unchanged. Argument handlers shorten the host code by managing device storage and transfers around the call. They do not remove the transfers or their cost. In this notebook, constructing fresh handlers for every call also repeats buffer allocation.

This interface is convenient for a self-contained operation. When several kernels repeatedly use the same arrays, the explicit persistent buffers of the first example allow data to remain on the GPU between launches.

**Execution status:** the notebook is supplied with cleared outputs. Its structure and Python syntax have been checked, but GPU execution has not been performed in the preparation environment. Run all cells on your assigned GPU to obtain the validation output and timings.

References: [PyCUDA argument-handler tutorial](https://documen.tician.de/pycuda/tutorial.html#shortcuts-for-explicit-memory-copies) and [PyCUDA driver implementation](https://github.com/inducer/pycuda/blob/main/pycuda/driver.py).